# MP4 から before / after 候補を選ぶ

この notebook は **動画探索と候補固定だけ**を行います。Lab解析は `video_selected_pair_lab.ipynb` に分離しました。

`RUN_SEARCH=False` なら既存の探索結果を再利用し、動画探索は実行しません。`RUN_SEARCH=True` のときだけ新規探索を実行します。


In [ ]:
from pathlib import Path
import json
import os

cwd = Path.cwd().resolve()
marker = Path('analysis/run_video_roi_search.py')
if (cwd / marker).is_file():
    REPO_ROOT = cwd
elif cwd.name == 'notebooks' and (cwd.parent / marker).is_file():
    REPO_ROOT = cwd.parent
    os.chdir(REPO_ROOT)
else:
    raise RuntimeError(f'ikiikimake のリポジトリ直下または notebooks/ から実行してください: current={cwd}')

print('repo root:', Path.cwd())

## 設定

探索済みの結果を見るだけなら `RUN_SEARCH=False` のままにします。新しく探索する場合だけ `True` にし、`OUTPUT` は新しい空フォルダ名にしてください。


In [ ]:
# ---- 実験条件 ----
VIDEO = Path('makeup.mp4')
OUTPUT = Path('outputs/makeup_video_search_final_window')

# メイク前 / 完成後として意味のある時間帯だけを探索する。
BEFORE_RANGE = (60.0, 180.0)       # 1:00〜3:00
AFTER_RANGE  = (1840.0, 1899.0)    # 30:40〜31:39

INTERVAL = 1.0
REFINE_INTERVAL = 0.5
TOP = 10

# 新しい時間帯を探索するので True。探索後に再利用するときだけ False にする。
RUN_SEARCH = True

# 目視で採用する順位。まだ選ばないなら None。
APPROVED_RANK = None


## 探索または既存結果の読み込み


In [ ]:
from collections import Counter
from analysis.run_video_roi_search import parse_args, run

matching_path = OUTPUT / 'matching.json'
manifest_path = OUTPUT / 'scan_manifest.json'

if RUN_SEARCH:
    if OUTPUT.exists() and any(OUTPUT.iterdir()):
        raise ValueError(f'RUN_SEARCH=True ですが OUTPUT が空ではありません: {OUTPUT}')
    argv = [
        '--video', str(VIDEO),
        '--output', str(OUTPUT),
        '--before-range', str(BEFORE_RANGE[0]), str(BEFORE_RANGE[1]),
        '--after-range', str(AFTER_RANGE[0]), str(AFTER_RANGE[1]),
        '--interval', str(INTERVAL),
        '--refine-interval', str(REFINE_INTERVAL),
        '--top', str(TOP),
    ]
    manifest = run(parse_args(argv))
else:
    if not matching_path.is_file() or not manifest_path.is_file():
        raise FileNotFoundError(
            f'RUN_SEARCH=False ですが既存探索結果がありません: {matching_path} / {manifest_path}'
        )
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    saved_ranges = manifest.get('selected_ranges')
    expected_ranges = {'before': list(BEFORE_RANGE), 'after': list(AFTER_RANGE)}
    if saved_ranges != expected_ranges:
        raise RuntimeError(
            '現在の BEFORE_RANGE / AFTER_RANGE と既存探索結果の範囲が一致しません。'
            f' current={expected_ranges} saved={saved_ranges}. '
            '新しい OUTPUT を指定して RUN_SEARCH=True で再探索してください。'
        )
    print('探索はスキップ。現在の範囲と一致する既存結果を使います:', OUTPUT)

if not matching_path.is_file() or not manifest_path.is_file():
    raise FileNotFoundError('探索結果ファイルが生成されていません。')

matching = json.loads(matching_path.read_text(encoding='utf-8'))
scan_manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
pairs = matching.get('ranked_pairs', [])

if not pairs:
    reasons = Counter(
        error
        for record in scan_manifest.get('records', [])
        for error in record.get('report', {}).get('errors', [])
    )
    print('比較候補は0件です。自動チェックで落ちた主な理由:')
    for reason, count in reasons.most_common(12):
        print(f'{count:3d}  {reason}')
    raise RuntimeError('新しい時間帯では eligible pair がありませんでした。')

print(f'{len(pairs)} candidates loaded')

## 候補の数値を確認

スコアは幾何差です。小さいほど撮影条件が近い候補で、美しさやメイク効果のスコアではありません。


In [ ]:
from IPython.display import HTML, display

columns = [
    ('rank', 'rank'), ('before_time', 'before_time'), ('after_time', 'after_time'),
    ('score', 'score'), ('yaw_gap', 'yaw_gap_degrees'), ('pitch_gap', 'pitch_gap_degrees'),
    ('roll_gap', 'roll_gap_degrees'), ('face_scale_ratio', 'face_scale_ratio'),
    ('roi_rms', 'roi_procrustes_rms'), ('eye_gap', 'eye_aperture_gap'), ('mouth_gap', 'mouth_opening_gap'),
]
headers = ''.join(f'<th>{label}</th>' for label, _ in columns)
rows = []
for rank, pair in enumerate(pairs, 1):
    values = {'rank': rank, 'before_time': pair['before_time'], 'after_time': pair['after_time'], 'score': pair['score'], **pair['terms']}
    cells = ''.join(
        f'<td>{values[key]:.4f}</td>' if isinstance(values[key], float) else f'<td>{values[key]}</td>'
        for _, key in columns
    )
    rows.append(f'<tr>{cells}</tr>')
display(HTML(f'<table><thead><tr>{headers}</tr></thead><tbody>{"".join(rows)}</tbody></table>'))

report = OUTPUT / 'report.html'
if not report.is_file():
    raise FileNotFoundError(report)
print('候補画像は report.html で確認:', report.resolve())

## 目視で選んだ候補を固定

`APPROVED_RANK` を設定して実行します。既に同じ候補が `selected_pair.json` に固定済みなら、内容を確認してそのまま再利用します。別の候補へ勝手に上書きはしません。


In [ ]:
import hashlib

if APPROVED_RANK is None:
    print('APPROVED_RANK=None: 候補固定はスキップしました。')
else:
    if isinstance(APPROVED_RANK, bool) or not isinstance(APPROVED_RANK, int):
        raise TypeError('APPROVED_RANK は整数で指定してください。')
    if not (1 <= APPROVED_RANK <= len(pairs)):
        raise ValueError(f'APPROVED_RANK は 1..{len(pairs)} の範囲です。')

    def sha256_file(path: Path) -> str:
        if not path.is_file():
            raise FileNotFoundError(path)
        digest = hashlib.sha256()
        with path.open('rb') as handle:
            for block in iter(lambda: handle.read(1024 * 1024), b''):
                digest.update(block)
        return digest.hexdigest()

    records = {record['frame_id']: record for record in scan_manifest['records']}
    pair = pairs[APPROVED_RANK - 1]
    selected = {'rank': APPROVED_RANK, 'score': pair['score'], 'before': {}, 'after': {}}
    for phase, key in (('before', 'before_id'), ('after', 'after_id')):
        record = records.get(pair[key])
        if record is None:
            raise RuntimeError(f'{phase} フレームが scan_manifest.json にありません: {pair[key]}')
        image_path = Path(record['image_path'])
        roi_dir = Path(record['roi_dir'])
        masks_path = roi_dir / 'roi_masks.npz'
        points_path = roi_dir / 'roi_points.json'
        overlay_path = roi_dir / 'roi_overlay.png'
        for required in (image_path, masks_path, points_path, overlay_path):
            if not required.is_file():
                raise FileNotFoundError(required)
        selected[phase] = {
            'frame_id': record['frame_id'],
            'timestamp_seconds': record['timestamp_seconds'],
            'image_path': str(image_path),
            'image_sha256': sha256_file(image_path),
            'roi_dir': str(roi_dir),
            'roi_masks_sha256': sha256_file(masks_path),
            'roi_points_sha256': sha256_file(points_path),
            'roi_overlay_sha256': sha256_file(overlay_path),
        }

    selected_path = OUTPUT / 'selected_pair.json'
    if selected_path.exists():
        existing = json.loads(selected_path.read_text(encoding='utf-8'))
        if existing != selected:
            raise FileExistsError(f'別内容の selected_pair.json が既にあります。上書きしません: {selected_path}')
        print('同じ候補が既に固定済みです:', selected_path)
    else:
        selected_path.write_text(json.dumps(selected, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
        print('固定しました:', selected_path)


## 次

Lab解析は `notebooks/video_selected_pair_lab.ipynb` を開き、同じ `OUTPUT` を指定して **Run All** してください。
この notebook 自体には大量の画像を埋め込まないので、保存時に巨大化しにくくしています。
